In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import dash
from dash import html, dcc
import plotly.graph_objects as go
import locale
import numpy as np
from functools import reduce
from src.data_preparation import concatenando_dados

In [3]:
# from src.data_preparation import gera_columns, prepara_dados_batismo, concatenando_dados, resultados_anuais
# from src.gera_figuras import gera_figura, gera_figura_atributo, gera_figura_batismo, gera_figura_media, prepara_dados_anuais_plot, prepara_dados_mensais_plot, ComparaDadosAnuais

In [4]:
locale.setlocale(locale.LC_ALL, 'pt_BR.UTF-8')

'pt_BR.UTF-8'

In [5]:
ws, hs = 900*1.5, 400*1.5

In [6]:
url_planilha_culto = 'https://docs.google.com/spreadsheets/d/1g3CmgKJvO4yLlqYFNyNreR3AFRUSnSIyDzc0ncuOPJI/export?format=csv'
url_planilha_culto_antigo = 'https://docs.google.com/spreadsheets/d/1IBiGdhHW7E9t5w9yU6VbT-6n58tcITaGyCGHddNeQmA/export?format=csv'
url_planilha_batismo = 'https://docs.google.com/spreadsheets/d/1B8vbwEEjdyRDWe_yKf1j1y-h5XM0bAwaqydNDRU4qLA/export?format=csv'

In [7]:
db =  pd.read_csv(url_planilha_batismo)

In [8]:
# db =  pd.read_csv('batismo.csv', sep=';')
# dbn = prepara_dados_batismo(db)

In [9]:
dc = pd.read_csv(url_planilha_culto)
do = pd.read_csv(url_planilha_culto_antigo)

# df = concatenando_dados(dc, do)

In [10]:
def concatenando_dados(dados1, dados2):
    
    dados = pd.concat([dados1, dados2])
    dados['Dia'] = pd.to_datetime(dados['Dia'],  format='%d/%m/%Y')
    dados['Dia'] = [dados['Dia'].iloc[i].strftime('%d/%m/%Y') for i in range(dados.shape[0])]
    dados.drop('Carimbo de data/hora', axis=1, inplace=True)

    dados["Dia"] = pd.to_datetime(dados["Dia"], format="%d/%m/%Y")

    dados_dia = (
        dados.groupby("Dia", as_index=False)
        .sum(numeric_only=True)
    )

    dados_dia.insert(1, "Período do culto", "Total")

    dados_dia["Dia"] = dados_dia["Dia"].dt.strftime("%d/%m/%Y")
    dados_dia['Dia'] = pd.to_datetime(dados_dia['Dia'],  format='%d/%m/%Y')

    dados = pd.concat([dados, dados_dia])
    dados.fillna(0, inplace=True)
    dados['Novos decididos'] = dados[['Número de conversões', 
                                      'Número de reconciliações']].sum(axis=1)
    
    dados['Percentual de visitantes'] =  np.round(
        np.divide(
            100 * np.array(dados['Número de visitantes']),
            np.array(dados['Número total de pessoas']),
            out=np.zeros_like(np.array(dados['Número de visitantes']), dtype=float),
            where=np.array(dados['Número total de pessoas']) != 0
        ),
        2
    )

    dados['Percentual de novo decididos'] =  np.round(
        np.divide(
            100 * np.array(dados['Novos decididos']),
            np.array(dados['Número total de pessoas']),
            out=np.zeros_like(np.array(dados['Novos decididos']), dtype=float),
            where=np.array(dados['Número total de pessoas']) != 0
        ),
        2
    )

    float_colunas = dados.select_dtypes(include=['float64']).columns.tolist()
    dados[float_colunas] = dados[float_colunas].fillna(value=0)

    dados.sort_values(by='Dia', inplace=True)
    dados.reset_index(drop=True, inplace=True)
   
    
    return dados


In [11]:
df = concatenando_dados(dc, do)

In [12]:
df.sort_values(by='Dia')

,Dia,Período do culto,Número total de pessoas,Número de visitantes,Número de conversões,Número de reconciliações,Número de carros,Novos decididos,Percentual de visitantes,Percentual de novo decididos
0,2024-01-07,Total,300,15,0,0,0.0,0,5.00,0.0
1,2024-01-07,Noite,180,12,0,0,0.0,0,6.67,0.0
2,2024-01-07,Manhã,120,3,0,0,0.0,0,2.50,0.0
3,2024-01-14,Noite,172,4,0,0,0.0,0,2.33,0.0
4,2024-01-14,Total,277,8,0,0,0.0,0,2.89,0.0
...,...,...,...,...,...,...,...,...,...,...
382,2026-06-28,Noite,212,9,0,0,74.0,0,4.25,0.0
384,2026-06-28,Total,409,15,0,0,147.0,0,3.67,0.0
386,2026-07-05,Noite,0,0,0,0,0.0,0,0.00,0.0
385,2026-07-05,Manhã,337,10,0,0,119.0,0,2.97,0.0


In [13]:
dados = df.copy()

In [14]:
df

,Dia,Período do culto,Número total de pessoas,Número de visitantes,Número de conversões,Número de reconciliações,Número de carros,Novos decididos,Percentual de visitantes,Percentual de novo decididos
0,2024-01-07,Total,300,15,0,0,0.0,0,5.00,0.0
1,2024-01-07,Noite,180,12,0,0,0.0,0,6.67,0.0
2,2024-01-07,Manhã,120,3,0,0,0.0,0,2.50,0.0
3,2024-01-14,Noite,172,4,0,0,0.0,0,2.33,0.0
4,2024-01-14,Total,277,8,0,0,0.0,0,2.89,0.0
...,...,...,...,...,...,...,...,...,...,...
383,2026-06-28,Manhã,197,6,0,0,73.0,0,3.05,0.0
384,2026-06-28,Total,409,15,0,0,147.0,0,3.67,0.0
385,2026-07-05,Manhã,337,10,0,0,119.0,0,2.97,0.0
386,2026-07-05,Noite,0,0,0,0,0.0,0,0.00,0.0


In [16]:


# fig = go.Figure()

# for periodo in df['Período do culto'].unique():
#     df = dados[dados["Período do culto"] == periodo].sort_values("Dia")

#     fig.add_trace(
#         go.Scatter(
#             x=df["Dia"],
#             y=df["Número total de pessoas"],
#             mode="lines+markers+text",
#             textposition='top center',
#             text=df["Número total de pessoas"],
#             name=periodo
#         )
#     )


dados_frequencia = [
        go.Scatter(
            x=df[df['Período do culto']==periodo]["Dia"],
            y=df[df['Período do culto']==periodo]["Número total de pessoas"],
            mode="lines+markers+text",
            textposition='top center',
            text=df["Número total de pessoas"],
            name=periodo
        ) for periodo in df['Período do culto'].unique()]

fig = go.Figure(dados_frequencia)

fig.update_layout(
    title="Frequência",
    xaxis_title="Data",
    yaxis_title="Número total de pessoas",
    template="plotly_dark",
    yaxis_rangemode='normal',
    width=ws,
    height=hs,
    hovermode="x unified"
)


fig.show()

In [17]:
df_filtrado = df.copy()

In [19]:

dados_carros = [
        go.Scatter(
            x=df_filtrado[df_filtrado['Período do culto']==periodo]["Dia"],
            y=df_filtrado[df_filtrado['Período do culto']==periodo]["Número de carros"],
            mode="lines+markers+text",
            textposition='top center',
            text=df_filtrado[df_filtrado['Período do culto']==periodo]["Número de carros"],
            name=periodo
        ) for periodo in df_filtrado['Período do culto'].unique()]

figPC = go.Figure(dados_carros)


figPC.update_layout(
    title={
        "text": "Número de carros no culto",
        "x": 0.5,
        "xanchor": "center",
        "font": {"size": 34}
    },
    xaxis_title="Data",
    xaxis_tickangle=-45,  
    yaxis_title="Número de carros",
    template="plotly_dark",
    yaxis_rangemode='normal',
    width=ws,
    height=hs,
    hovermode="x unified"
)


In [ ]:
df.describe()

In [ ]:
dados = [go.Scatter(x=df['Dia'], 
                    y=df['Número total de pessoas'], 
                    name=f'{atr}',
                    mode='lines+markers+text', 
                    textposition='top center',
                    text=df[f'{atr}']) for atr in df['Período do culto']]

In [ ]:
data.append(go.Scatter(x=dados['Dia'], 
                            y=dados[f'{atributo} {_per}'], 
                            name=f'{atributo} {_per}',
                            mode='lines+markers+text', 
                            textposition='top center',
                            text=dados[f'{atributo} {_per}']))

In [ ]:
def gera_figura_atributo(dados, 
                         atributo, 
                         ws, 
                         hs, 
                         _period, 
                         _colunay='Número de pessoas'):

    data = []
    _ynote = []

    for _per in _period:
        data.append(go.Scatter(x=dados['Dia'], 
                            y=dados[f'{atributo} {_per}'], 
                            name=f'{atributo} {_per}',
                            mode='lines+markers+text', 
                            textposition='top center',
                            text=dados[f'{atributo} {_per}']))
        
        _ynote.append(np.mean(dados[f'{atributo} {_per}']))
    
    fig = go.Figure(data)
    fig.update_layout(title=f'Gráfico de {atributo.title()}',
                    xaxis_title='Dia',
                    yaxis_title=_colunay,
                    yaxis_rangemode='normal',
                    width=ws,
                    height=hs,
                    template='plotly_dark')

    return fig.show()


In [ ]:
gera_figura_atributo(df, 'Frequência', ws, hs, ['manhã', 'noite', 'total'])

In [ ]:
df.sort_values(by='Dia')

In [ ]:
dados = pd.concat([dados1, dados2])

In [ ]:
dc

In [ ]:
df

In [ ]:
dc.dtypes


In [ ]:
dc

In [ ]:
df['Dia'] = pd.to_datetime(df['Dia'],format='%d/%m/%Y')
df = gera_columns(df)
float_colunas = df.select_dtypes(include=['float64']).columns.tolist()
df[float_colunas] = df[float_colunas].fillna(value=0)
df[float_colunas] = np.around(df[float_colunas], 3)

In [ ]:
df.columns

In [ ]:
df = pd.con

In [ ]:
df

In [ ]:
dy = resultados_anuais(df, db)

In [ ]:
dados_absolutos, dados_percentuais = prepara_dados_anuais_plot(dy)

In [ ]:
dados_frequencia_mensal, dados_visitantes_mensais, dados_media_novos_decididos = prepara_dados_mensais_plot(df)

In [ ]:
figM = gera_figura(df, 'manhã', ws, hs)
figN = gera_figura(df, 'noite', ws, hs)
figT = gera_figura(df, 'total', ws, hs)

In [ ]:
figM.show()

In [ ]:
figFq = gera_figura_atributo(df, 'Frequência', ws, hs, ['manhã', 'noite', 'total'])
figVt = gera_figura_atributo(df, 'Visitantes', ws, hs, ['manhã', 'noite', 'total'])
figCv = gera_figura_atributo(df, 'Conversões', ws, hs, ['manhã', 'noite', 'total'])
figRc = gera_figura_atributo(df, 'Reconciliações', ws, hs, ['manhã', 'noite', 'total'])
figVF = gera_figura_atributo(df, 'Visitantes/Frequência', ws, hs, ['manhã', 'noite', 'total'])
figNF = gera_figura_atributo(df, 'Novos Decididos/Frequência', ws, hs, ['manhã', 'noite', 'total'])

In [ ]:
figFq.show()

In [ ]:
df.dtypes

In [ ]:
figbt = gera_figura_batismo(db, 'Data', 'Quantidade', 'dia', ws, hs)
figbta = gera_figura_batismo(dbn, 'Ano', 'soma', 'ano', ws, hs)

In [ ]:
fig_media_f = gera_figura_media(dados_frequencia_mensal, 
                                'Média mensal da frequência', 
                                ws, 
                                hs)

fig_media_visitante = gera_figura_media(dados_visitantes_mensais, 
                                        'Média mensal do número de visitantes',
                                        ws, 
                                        hs)

fig_media_nd = gera_figura_media(dados_media_novos_decididos, 
                                        'Média mensal do número de novos decididos', 
                                        ws, 
                                        hs)

In [ ]:
figA = gera_figura_media(dados_absolutos, 'Dados anuais', ws, hs)
figPA = gera_figura_media(dados_percentuais, 'Percentual de n° conversões por n° de batismos', ws, hs)

In [ ]:
colunas_compara = ['Dia', 
 'Frequência manhã', 'Visitantes manhã', 'ND manhã',
 'Frequência noite', 'Visitantes noite', 'ND noite', 
 'Frequência total', 'Visitantes total', 'ND total'
 ]

CDA = ComparaDadosAnuais(df,
                         colunas_compara)

fator = 1.8

In [ ]:
df[colunas_compara]

In [ ]:
figAFM = CDA.plot_grafico('Id', 'Frequência manhã', ws, fator*hs)
figAVM = CDA.plot_grafico('Id', 'Visitantes manhã', ws, fator*hs)
figANDM = CDA.plot_grafico('Id', 'ND manhã', ws, fator*hs)
figAFN = CDA.plot_grafico('Id', 'Frequência noite', ws, fator*hs)
figAVN = CDA.plot_grafico('Id', 'Visitantes noite', ws, fator*hs)
figANDN = CDA.plot_grafico('Id', 'ND noite', ws, fator*hs)
figAFT = CDA.plot_grafico('Id', 'Frequência total', ws, fator*hs)
figAVT = CDA.plot_grafico('Id', 'Visitantes total', ws, fator*hs)
figANDT = CDA.plot_grafico('Id', 'ND total', ws, fator*hs)


In [ ]:
app = dash.Dash(__name__)

app.layout = html.Div([
        # dcc.Graph(id='graph1', figure=figM, style={'width': '100%'}),
        # dcc.Graph(id='graph2', figure=figN, style={'width': '100%'}),
        # dcc.Graph(id='graph3', figure=figT, style={'width': '100%'}),

        dcc.Graph(id='graph4', figure=figFq, style={'width': '100%'}),
        dcc.Graph(id='graph5', figure=figVt, style={'width': '100%'}),
        dcc.Graph(id='graph6', figure=figCv, style={'width': '100%'}),
        dcc.Graph(id='graph7', figure=figRc, style={'width': '100%'}),

        dcc.Graph(id='graph8', figure=figVF, style={'width': '100%'}),
        dcc.Graph(id='graph9', figure=figNF, style={'width': '100%'}),

        dcc.Graph(id='graph10', figure=fig_media_f, style={'width': '100%'}),
        dcc.Graph(id='graph11', figure=fig_media_visitante, style={'width': '100%'}),
        dcc.Graph(id='graph12', figure= fig_media_nd, style={'width': '100%'}),
        
        dcc.Graph(id='graph13', figure=figbt, style={'width': '100%'}),
        dcc.Graph(id='graph14', figure=figbta, style={'width':'100%'}),

        dcc.Graph(id='graph15', figure=figA, style={'width': '100%'}),
        dcc.Graph(id='graph16', figure=figPA, style={'width':'100%'}),

        dcc.Graph(id='graph17', figure=figAFT, style={'width': '100%'}),
        dcc.Graph(id='graph18', figure=figAVT, style={'width':'100%'}),
        dcc.Graph(id='graph19', figure=figANDT, style={'width':'100%'}),

        dcc.Graph(id='graph20', figure=figAFM, style={'width': '100%'}),
        dcc.Graph(id='graph21', figure=figAVM, style={'width':'100%'}),
        dcc.Graph(id='graph22', figure=figANDM, style={'width':'100%'}),

        dcc.Graph(id='graph23', figure=figAFN, style={'width': '100%'}),
        dcc.Graph(id='graph24', figure=figAVN, style={'width':'100%'}),
        dcc.Graph(id='graph35', figure=figANDN, style={'width':'100%'}),
        
 

    ])



if __name__ == '__main__':
    app.run_server(debug=True)